In [ ]:
import importlib.util
import subprocess
import sys

required = ["pandas", "numpy", "sklearn"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent))

import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from src.evaluate import load_data, cv_rmsle
from src.features import build_features

train, test = load_data()

X = build_features(train)
y = train["count"]

print("Training data:", X.shape)
print("Target:", y.shape)

Training data: (10886, 67)
Target: (10886,)


In [6]:
from sklearn.model_selection import ParameterGrid

dt_params = {
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

dt_results = []

for params in ParameterGrid(dt_params):

    model = DecisionTreeRegressor(
        random_state=42,
        **params
    )

    score = cv_rmsle(
        model,
        X,
        y,
        log_target=True
    )

    dt_results.append({
        **params,
        "cv_rmsle": score
    })

    print(params, "->", round(score, 4))

dt_results_df = pd.DataFrame(dt_results)

dt_results_df = dt_results_df.sort_values(
    "cv_rmsle"
).reset_index(drop=True)

dt_results_df.head(10)

{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2} -> 0.5684
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5} -> 0.5674
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10} -> 0.564
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2} -> 0.567
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5} -> 0.5666
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10} -> 0.5637
{'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2} -> 0.5615
{'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 5} -> 0.5615
{'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 10} -> 0.5615
{'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2} -> 0.4576
{'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 5} -> 0.4516
{'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 10} -> 0.4417
{'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 2} -> 0.4496
{'max_depth': 15, 'min_

,max_depth,min_samples_leaf,min_samples_split,cv_rmsle
0,NaN,5,5,0.410521
1,NaN,5,10,0.410521
2,NaN,5,2,0.410521
3,NaN,2,10,0.414043
4,20.0,5,10,0.414403
5,20.0,5,2,0.414403
6,20.0,5,5,0.414403
7,20.0,2,10,0.415530
8,NaN,1,10,0.416884
9,20.0,1,10,0.421705


In [10]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 15, 25],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 1.0]
}

rf_results = []

for params in ParameterGrid(rf_params):

    model = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **params
    )

    score = cv_rmsle(
        model,
        X,
        y,
        log_target=True
    )

    rf_results.append({
        **params,
        "cv_rmsle": score
    })

    print(params, "->", round(score, 4))

rf_results_df = pd.DataFrame(rf_results)

rf_results_df = rf_results_df.sort_values(
    "cv_rmsle"
).reset_index(drop=True)

rf_results_df.head(10)

{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} -> 0.4028
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} -> 0.4011
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100} -> 0.4107
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200} -> 0.4109
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100} -> 0.4333
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200} -> 0.4333
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100} -> 0.434
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200} -> 0.4325
{'max_depth': None, 'max_

,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,cv_rmsle
0,NaN,1.0,1,2,200,0.321353
1,NaN,1.0,1,5,200,0.321450
2,NaN,1.0,1,5,100,0.322307
3,NaN,1.0,1,2,100,0.322612
4,25.0,1.0,1,2,200,0.322781
5,25.0,1.0,1,5,200,0.322960
6,25.0,1.0,1,5,100,0.323708
7,25.0,1.0,1,2,100,0.323906
8,NaN,1.0,2,2,200,0.325704
9,NaN,1.0,2,5,200,0.325707


In [11]:
print("Best Random Forest parameters:")
print(rf_results_df.iloc[0])

Best Random Forest parameters:
max_depth                 NaN
max_features              1.0
min_samples_leaf            1
min_samples_split           2
n_estimators              200
cv_rmsle             0.321353
Name: 0, dtype: object


In [16]:
gb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

gb_results = []

for params in ParameterGrid(gb_params):

    model = GradientBoostingRegressor(
        random_state=42,
        **params
    )

    score = cv_rmsle(
        model,
        X,
        y,
        log_target=True
    )

    gb_results.append({
        **params,
        "cv_rmsle": score
    })

    print(params, "->", round(score, 4))

gb_results_df = pd.DataFrame(gb_results)

gb_results_df = gb_results_df.sort_values(
    "cv_rmsle"
).reset_index(drop=True)

gb_results_df.head(10)

{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} -> 0.8264
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} -> 0.6322
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300} -> 0.538
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100} -> 0.8264
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200} -> 0.6322
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300} -> 0.538
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100} -> 0.8264
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200} -> 0.6322
{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf'

,learning_rate,max_depth,min_samples_leaf,min_samples_split,n_estimators,cv_rmsle
0,0.1,4,2,5,300,0.306316
1,0.1,4,1,5,300,0.306630
2,0.1,4,1,2,300,0.307067
3,0.1,4,2,2,300,0.307361
4,0.1,4,2,5,200,0.320859
5,0.1,4,1,2,200,0.320917
6,0.1,4,2,2,200,0.321415
7,0.1,4,1,5,200,0.321591
8,0.1,3,1,5,300,0.324004
9,0.1,3,1,2,300,0.324353


In [17]:
if "gb_results_df" in globals():
    print("Best Gradient Boosting parameters:")
    print(gb_results_df.iloc[0])
else:
    print("Run the Gradient Boosting tuning cell first.")


Best Gradient Boosting parameters:
learning_rate          0.100000
max_depth              4.000000
min_samples_leaf       2.000000
min_samples_split      5.000000
n_estimators         300.000000
cv_rmsle               0.306316
Name: 0, dtype: float64


In [18]:
from pathlib import Path

Path("../results").mkdir(exist_ok=True)

dt_results_df.to_csv("../results/decision_tree_tuning.csv", index=False)
rf_results_df.to_csv("../results/random_forest_tuning.csv", index=False)
gb_results_df.to_csv("../results/gradient_boosting_tuning.csv", index=False)

print("Tuning results saved successfully.")

Tuning results saved successfully.


In [ ]:
print("Best Decision Tree parameters:")
print(dt_results_df.iloc[0])